# Flash-attention before/after speed benchmark (T4)

Times each model with **eager / sdpa / flash_attention_2** to decide whether flash-attention should be the default. Thin runner: clone → install (incl. flash-attn) → run `scripts/bench_attention.py`.

Runtime → Change runtime type → **T4 GPU**, then Run all.


## 1. Setup (clone + install + flash-attn)

In [4]:
%cd /content
![ -d OCR ] || git clone https://github.com/SangbumChoi/OCR.git


/content


In [5]:
%cd /content/OCR
!git checkout claude/new-session-w79q0i && git pull --ff-only

/content/OCR
Branch 'claude/new-session-w79q0i' set up to track remote branch 'claude/new-session-w79q0i' from 'origin'.
Switched to a new branch 'claude/new-session-w79q0i'
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (7/7), done.
remote: Total 8 (delta 6), reused 7 (delta 6), pack-reused 1 (from 1)
Unpacking objects: 100% (8/8), 1.71 KiB | 250.00 KiB/s, done.
From https://github.com/SangbumChoi/OCR
   30bd772..756cfb9  claude/new-session-w79q0i -> origin/claude/new-session-w79q0i
   30bd772..756cfb9  main       -> origin/main
Updating 30bd772..756cfb9
Fast-forward
 notebooks/flash_attention_benchmark.ipynb | 32 ++++++++++++++++++++++++-------
 results/matrix_capability.json            | 11 ++++++++---
 results/matrix_capability.md              | 10 +++++-----
 3 files changed, 38 insertions(+), 15 deletions(-)


In [6]:
!pip -q install -e '.[models]'
# flash-attn wheel (may take a few min); if it fails, eager/sdpa rows still run
!pip -q install flash-attn --no-build-isolation || echo 'flash-attn install failed; that row will be skipped'

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 43.8 MB/s eta 0:00:00
  Building editable for docvlm-eval (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for flash-attn
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (flash-attn)
flash-attn install failed; that row will be skipped


## 2. Benchmark a few representative models

In [7]:
%cd /content/OCR
!python scripts/make_capability_probe.py
!python scripts/bench_attention.py --models smolvlm-500m llava-ov-0.5b internvl2_5-1b --device cuda

/content/OCR
[done] 6 capability samples -> data/benchmarks/capability_probe/capability.jsonl
   text-recognition     metric=anls         gold=INV-2025-0042
   kie-localized        metric=anls         gold=Acme Corporation
   integrative-sum      metric=relaxed_acc  gold=145.50
   integrative-rel      metric=exact        gold=Gadget B
   chart-dependent      metric=relaxed_acc  gold=70
   location-grounding   metric=grounding    gold=40,334,222,355;820,600
2026-06-18 02:12:49.921856: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
processor_config.json: 100% 68.0/68.0 [00:00<00:00, 540kB/s]
chat_template.json: 100% 429/429 [00:00<00:00, 2.43MB/s]
preprocessor_config.json: 100% 486/486 [00:00<00:00, 3.93MB/s]
tokenizer_config.json: 28.2kB [00:00, 

## 3. Result

In [8]:
print(open('/content/OCR/results/attention_benchmark.md').read())

# Flash-attention speed benchmark

Benchmark: `data/benchmarks/capability_probe/capability.jsonl` · device: cuda · dtype: bfloat16

| model          | attn              | load(s) | avg lat(s) | p90(s) | peak GPU(MB) | status            |
| -------------- | ----------------- | ------- | ---------- | ------ | ------------ | ----------------- |
| smolvlm-500m   | eager             | 33.28   | 2.977      | 3.79   | 2952.6       | ok                |
| smolvlm-500m   | sdpa              | 2.98    | 2.863      | 3.582  | 2952.8       | ok                |
| smolvlm-500m   | flash_attention_2 | -       | -          | -      | -            | FAIL: ImportError |
| llava-ov-0.5b  | eager             | 28.85   | 5.595      | 6.5    | 5174.1       | ok                |
| llava-ov-0.5b  | sdpa              | 6.62    | 5.701      | 6.051  | 5750.4       | ok                |
| llava-ov-0.5b  | flash_attention_2 | -       | -          | -      | -            | FAIL: ImportError |
| internvl2_5-1b | e

## 4. If flash wins
Re-run the full comparison with `--attn flash_attention_2`:
`python scripts/run_matrix.py --all --benchmark data/benchmarks/capability_probe/capability.jsonl --device cuda --attn flash_attention_2`, or change the adapter default in `src/docvlm_eval/models/base.py` (`resolve_attn`).